# Independent Repeated Generations (k = 3) on Kaggle GPU — Local Ollama Models

Addresses **R1C3 / R2C14 / R2C15**: runs **all 50 tasks × 6 core prompt conditions × k = 3 independent repeats = 900 calls per model–task pair** on Kaggle GPU using the exact same protocol as the original local experiments (`temperature = 0.2`, `num_predict = 2048`, `concurrency = 1`, 1.0s pacing).

**Core 6 Prompt Conditions (`category_id`):**
- `1`: Neutral + No Role + Plaintext *(baseline)*
- `2`: Neutral + No Role + JSON *(Format: JSON vs Plaintext)*
- `3`: Neutral + No Role + XML *(Format: XML vs Plaintext)*
- `4`: Neutral + Expert/Developer Role + Plaintext *(Role vs No Role)*
- `7`: Polite + No Role + Plaintext *(Tone: Polite vs Neutral)*
- `13`: Aggressive + No Role + Plaintext *(Tone: Aggressive vs Neutral)*

**Kaggle Setup:**
1. **Settings → Internet**: ON (to install Ollama and pull models).
2. **Accelerator**: GPU P100 (matches original study hardware) or GPU T4 x2.
3. **Input Dataset**: Add `jakenty1/vnmathproblems2026` (contains `problems.csv` and `problem.math.csv`).
4. **Resume-safe**: Every record is flushed immediately to `/kaggle/working/*.jsonl`. Re-running skips completed `(task_id, category_id, rep_id)` keys.


In [ ]:
# === CONFIGURATION ===
# Recommended representative local models (run one per Kaggle session for clean 12h budgeting):
#   "qwen2.5:7b", "llama3.1:8b", "gemma2:9b", "nemotron-3-nano:4b", "deepseek-coder:6.7b-instruct", "mistral:7b-instruct"
MODEL_CHOICE = "qwen2.5:7b"
TASK_TYPE = "both"  # "code", "math", or "both"
START_PROBLEM = 1
END_PROBLEM = 50
CORE_CATEGORY_IDS = ["1", "2", "3", "4", "7", "13"]
REPEATS = 3

# Decoding & runtime settings (identical to original study Section 3.5)
TEMPERATURE = 0.2
NUM_PREDICT = 2048
INTER_CALL_DELAY = 1.0
HTTP_TIMEOUT = 300
EVAL_TIMEOUT = 10

PYTHON_SYSTEM_PROMPT = "Output only the code, no explanations."
MATH_SYSTEM_PROMPT = "Output only the final result, no explanations."

# Optional: if resuming from a previous Kaggle notebook output attached as an input dataset,
# set RESUME_FROM_DIR to that directory path (e.g. "/kaggle/input/my-prev-run"), else None.
RESUME_FROM_DIR = None


In [ ]:
# Verify Kaggle input files
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith(('.csv', '.jsonl')):
            print(os.path.join(dirname, filename))


In [ ]:
# Install zstd and Ollama
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
# Start Ollama server in background
import subprocess, time, requests

os.environ["OLLAMA_HOST"] = "0.0.0.0"
os.environ["OLLAMA_ORIGINS"] = "*"

subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("ollama_stdout.log", "w"),
    stderr=open("ollama_stderr.log", "w"),
)

for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print(f"Ollama server ready after {i + 1}s")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("ERROR: Ollama server failed to start.")
    !tail -n 20 ollama_stderr.log


In [ ]:
# Pull the target model
print(f"Pulling {MODEL_CHOICE}...")
!ollama pull {MODEL_CHOICE}
!ollama list


In [ ]:
# === CORE RUNNER & EVALUATORS (Code + Math, saving full raw response_text) ===
import ast
import csv
import json
import math
import multiprocessing
import operator
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path

CATEGORY_LABELS = {
    "1": "neutral_norole_plaintext",
    "2": "neutral_norole_json",
    "3": "neutral_norole_xml",
    "4": "neutral_role_plaintext",
    "7": "polite_norole_plaintext",
    "13": "aggressive_norole_plaintext",
}

def find_csv(filename: str) -> str:
    candidates = [
        f"/kaggle/input/datasets/jakenty1/vnmathproblems2026/{filename}",
        f"/kaggle/input/vnmathproblems2026/{filename}",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    for dirname, _, filenames in os.walk("/kaggle/input"):
        if filename in filenames:
            return os.path.join(dirname, filename)
    raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")

def call_ollama(model: str, system_prompt: str, user_prompt: str):
    start = time.time()
    payload = {
        "model": model,
        "prompt": user_prompt,
        "stream": False,
        "options": {"temperature": TEMPERATURE, "num_predict": NUM_PREDICT},
    }
    if system_prompt:
        payload["system"] = system_prompt
    try:
        r = requests.post("http://localhost:11434/api/generate", json=payload, timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        data = r.json()
        return data.get("response", ""), {
            "duration_seconds": round(time.time() - start, 3),
            "prompt_tokens": data.get("prompt_eval_count", 0),
            "completion_tokens": data.get("eval_count", 0),
            "done_reason": data.get("done_reason", ""),
            "success": True,
            "error": None,
        }
    except Exception as e:
        return "", {
            "duration_seconds": round(time.time() - start, 3),
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "done_reason": "",
            "success": False,
            "error": str(e),
        }

# --- Python Evaluator ---
def extract_python_code(response: str) -> str:
    matches = re.findall(r"```python\s*\n(.*?)```", response, re.DOTALL | re.IGNORECASE)
    if matches:
        return matches[-1].strip()
    matches = re.findall(r"```\s*\n(.*?)```", response, re.DOTALL)
    if matches:
        return matches[-1].strip()
    return response.strip()

def is_python_code(code: str) -> bool:
    try:
        ast.parse(code)
        return True
    except SyntaxError:
        return False

def _run_test_worker(code, test, result_queue):
    try:
        local_ns = {}
        exec(code + "\n" + test, {}, local_ns)
        result_queue.put((True, None))
    except Exception as e:
        result_queue.put((False, str(e)))

def run_test_with_timeout(code, test, timeout=EVAL_TIMEOUT):
    q = multiprocessing.Queue()
    p = multiprocessing.Process(target=_run_test_worker, args=(code, test, q))
    p.start()
    p.join(timeout)
    if p.is_alive():
        p.terminate()
        p.join()
        return False, f"Test timed out after {timeout}s"
    if not q.empty():
        return q.get()
    return False, "Test process exited unexpectedly"

def evaluate_python(response_text: str, test_list: list[str]):
    code = extract_python_code(response_text)
    if not code:
        return code, {"passed": False, "tests_passed": 0, "tests_total": len(test_list), "errors": ["No code extracted"]}
    if not is_python_code(code):
        return code, {"passed": False, "tests_passed": 0, "tests_total": len(test_list), "errors": ["Response is not valid Python"]}
    passed_count = 0
    errors = []
    for test in test_list:
        ok, err = run_test_with_timeout(code, test, timeout=EVAL_TIMEOUT)
        if ok:
            passed_count += 1
        elif err:
            errors.append(f"Test failed: {test} | Error: {err}")
    return code, {
        "passed": passed_count == len(test_list) and len(test_list) > 0,
        "tests_passed": passed_count,
        "tests_total": len(test_list),
        "errors": errors,
    }

# --- Math Evaluator (identical to original study) ---
def extract_latex_command_argument(text: str, command: str):
    marker = f"\\{command}" + "{"
    start = text.rfind(marker)
    if start < 0:
        return None
    idx = start + len(marker)
    depth = 1
    chars = []
    while idx < len(text):
        ch = text[idx]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(chars)
        chars.append(ch)
        idx += 1
    return None

def normalize_response_text(response: str) -> str:
    text = str(response).strip()
    text = re.sub(r"```(?:text|json|python)?\s*", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "").strip()
    text = text.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    boxed = extract_latex_command_argument(text, "boxed")
    if boxed is not None:
        text = boxed.strip()
    for pattern in [r"(?:final answer|answer|result)\s*(?:is|:)?\s*(.+)$", r"(?:therefore|so),?\s*(.+)$"]:
        matches = re.findall(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if matches:
            text = matches[-1].strip()
            break
    lines = [ln.strip() for ln in text.splitlines() if ln.strip() and ln.strip() not in {"\\[", "\\]", "$$", "$"}]
    if lines:
        text = lines[-1]
    return text.strip().strip(".")

def _replace_latex_frac(text: str) -> str:
    pattern = re.compile(r"\\(?:dfrac|tfrac|frac)\s*\{([^{}]+)\}\s*\{([^{}]+)\}")
    prev = None
    while prev != text:
        prev = text
        text = pattern.sub(r"(\1)/(\2)", text)
    return text

def normalize_math_expression(value) -> str:
    text = str(value).strip().strip("$ ")
    for a, b in [("\\(", ""), ("\\)", ""), ("\\[", ""), ("\\]", ""), ("\\left", ""), ("\\right", ""), ("{,}", ","), (",", ""), ("−", "-"), ("–", "-"), ("—", "-"), ("×", "*"), ("·", "*"), ("\\cdot", "*"), ("^", "**")]:
        text = text.replace(a, b)
    text = _replace_latex_frac(text)
    text = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", text)
    text = re.sub(r"√\s*([0-9.]+)", r"sqrt(\1)", text)
    return text.strip()

_BIN = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow}
_UN = {ast.UAdd: operator.pos, ast.USub: operator.neg}

def _safe_eval(node):
    if isinstance(node, ast.Expression): return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _BIN: return _BIN[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _UN: return _UN[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Name) and node.id in {"pi": math.pi, "e": math.e}: return float({"pi": math.pi, "e": math.e}[node.id])
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "sqrt" and len(node.args) == 1:
        return float(math.sqrt(_safe_eval(node.args[0])))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

def parse_math_value(value) -> float:
    if isinstance(value, (int, float)): return float(value)
    text = normalize_math_expression(value)
    if not text: raise ValueError("empty")
    pct = text.endswith("%")
    if pct: text = text[:-1].strip()
    if re.fullmatch(r"[-+]?\d+", text):
        res = float(int(text, 10))
        return res / 100 if pct else res
    res = float(_safe_eval(ast.parse(text, mode="eval")))
    return res / 100 if pct else res

def extract_numbers(text: str):
    norm = normalize_response_text(text)
    cands = [norm]
    if "=" in norm: cands.append(norm.split("=")[-1].strip())
    for c in cands:
        try: return [parse_math_value(c)]
        except Exception: pass
    vals = []
    pat = re.compile(r"\\(?:dfrac|tfrac|frac)\s*\{[^{}]+\}\s*\{[^{}]+\}|\\sqrt\s*\{[^{}]+\}|sqrt\s*\([^()]+\)|√\s*[0-9.]+|[-+]?\d[\d,]*(?:\.\d+)?\s*/\s*[-+]?\d[\d,]*(?:\.\d+)?|[-+]?\d[\d,]*(?:\.\d+)?\s*%?|[-+]?\.\d+\s*%?")
    for m in pat.finditer(norm):
        try: vals.append(parse_math_value(m.group(0)))
        except Exception: continue
    return vals

def evaluate_math(response_text: str, expected_raw: str, tolerance: float = 1e-6):
    try:
        loaded = json.loads(expected_raw)
    except Exception:
        loaded = expected_raw
    expected_vals = [parse_math_value(x) for x in loaded] if isinstance(loaded, list) else [parse_math_value(loaded)]
    norm = normalize_response_text(response_text)
    nums = extract_numbers(response_text)
    errors = []
    if len(nums) < len(expected_vals):
        errors.append(f"Expected {len(expected_vals)} numeric value(s), extracted {len(nums)}: {nums}")
        return norm, {"passed": False, "values_passed": 0, "values_total": len(expected_vals), "expected_answer": expected_vals, "extracted_values": nums, "errors": errors}
    cand = nums[-len(expected_vals):]
    ok = 0
    for a, e in zip(cand, expected_vals):
        if abs(a - e) <= tolerance: ok += 1
        else: errors.append(f"Expected {e}, got {a}")
    return norm, {"passed": ok == len(expected_vals), "values_passed": ok, "values_total": len(expected_vals), "expected_answer": expected_vals, "extracted_values": cand, "errors": errors}

# --- Resume-safe JSONL helpers ---
def make_run_key(task_id: str, category_id: str, rep_id: int) -> str:
    return f"{task_id}::{category_id}::rep{rep_id}"

def load_completed_keys(path: str) -> set[str]:
    done = set()
    if not os.path.exists(path):
        return done
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                if rec.get("error") is None and rec.get("row_key"):
                    done.add(str(rec["row_key"]))
            except Exception:
                continue
    return done

def append_jsonl(path: str, record: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


In [ ]:
# === EXECUTE REPEATED RUNS (50 tasks x 6 core conditions x k=3 repeats) ===
model_slug = re.sub(r"[^A-Za-z0-9_.-]+", "-", MODEL_CHOICE).strip("-").lower()
task_types_to_run = ["code", "math"] if TASK_TYPE == "both" else [TASK_TYPE]
output_files = []

for ttype in task_types_to_run:
    csv_name = "problems.csv" if ttype == "code" else "problem.math.csv"
    csv_path = find_csv(csv_name)
    sys_prompt = PYTHON_SYSTEM_PROMPT if ttype == "code" else MATH_SYSTEM_PROMPT
    out_filename = f"/kaggle/working/{model_slug}.{ttype}.repeated_k{REPEATS}.jsonl"
    output_files.append(out_filename)

    # If resuming from a prior Kaggle dataset output, copy it into /kaggle/working first
    if RESUME_FROM_DIR and not os.path.exists(out_filename):
        prev = os.path.join(RESUME_FROM_DIR, os.path.basename(out_filename))
        if os.path.exists(prev):
            shutil.copy2(prev, out_filename)
            print(f"Copied prior progress from {prev}")

    completed = load_completed_keys(out_filename)
    rows = []
    with open(csv_path, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            tid_num = int(re.sub(r"[^0-9]", "", str(row["task_id"])) or 0)
            if START_PROBLEM <= tid_num <= END_PROBLEM and str(row["category_id"]) in CORE_CATEGORY_IDS:
                rows.append(row)

    # Outer loop over rep_id so repetitions are separated in time
    schedule = [(rep_id, row) for rep_id in range(1, REPEATS + 1) for row in rows]
    remaining = [
        (rep_id, row) for rep_id, row in schedule
        if make_run_key(str(row["task_id"]), str(row["category_id"]), rep_id) not in completed
    ]
    print(f"\n=== [{ttype.upper()}][{MODEL_CHOICE}] total={len(schedule)}, done={len(schedule)-len(remaining)}, remaining={len(remaining)} ===")

    for idx, (rep_id, row) in enumerate(remaining, start=1):
        tid = str(row["task_id"])
        cid = str(row["category_id"])
        rkey = make_run_key(tid, cid, rep_id)
        response_text, meta = call_ollama(MODEL_CHOICE, sys_prompt, row["prompt"].strip())

        if not meta["success"]:
            print(f"[{idx}/{len(remaining)}] rep={rep_id} task={tid} cat={cid} -> ERROR: {meta['error']}", flush=True)
            continue

        if ttype == "code":
            tests = json.loads(row.get("test_cases", "[]")) + json.loads(row.get("challenge_test_list", "[]"))
            extracted, eval_res = evaluate_python(response_text, tests)
            extra = {"extracted_code": extracted, "tests_passed": eval_res["tests_passed"], "tests_total": eval_res["tests_total"]}
        else:
            norm, eval_res = evaluate_math(response_text, row.get("expected_answer", ""))
            extra = {"normalized_response": norm, "expected_answer": row.get("expected_answer", "")}

        record = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "dataset": "python" if ttype == "code" else "math",
            "model": MODEL_CHOICE,
            "model_label": model_slug,
            "rep_id": rep_id,
            "row_key": rkey,
            "task_id": tid,
            "category_id": cid,
            "category_label": CATEGORY_LABELS.get(cid, ""),
            "passed": bool(eval_res["passed"]),
            "duration_seconds": meta["duration_seconds"],
            "prompt_tokens": meta["prompt_tokens"],
            "completion_tokens": meta["completion_tokens"],
            "done_reason": meta["done_reason"],
            "error": meta["error"],
            "eval_result": eval_res,
            "response_text": response_text,  # FULL RAW RESPONSE saved for both code & math
            **extra,
        }
        append_jsonl(out_filename, record)

        if idx % 10 == 0 or idx == len(remaining):
            print(
                f"[{ttype}][{idx}/{len(remaining)}] rep={rep_id} task={tid} cat={cid} "
                f"{'PASS' if record['passed'] else 'FAIL'} ({meta['duration_seconds']}s, {meta['completion_tokens']} tok)",
                flush=True,
            )
        if INTER_CALL_DELAY > 0:
            time.sleep(INTER_CALL_DELAY)

    print(f"Saved -> {out_filename}")


In [ ]:
# === QUICK SUMMARY: Within-Cell Variability & Per-Condition Pass Rates ===
import pandas as pd

for path in output_files:
    if not os.path.exists(path):
        continue
    df = pd.read_json(path, lines=True)
    print(f"\n=== Summary: {os.path.basename(path)} (n={len(df)}) ===")
    cell = df.groupby(["task_id", "category_id"])["passed"].agg(["count", "mean", "nunique"])
    flip_rate = (cell["nunique"] > 1).mean() * 100
    print(f"Cells with all {REPEATS} repeats: {(cell['count'] == REPEATS).sum()} / {len(cell)}")
    print(f"Within-cell flip (disagreement) rate: {flip_rate:.2f}%")
    by_cat = df.groupby(["category_id", "category_label"])["passed"].agg(["count", "mean"])
    by_cat["pass_pct"] = (by_cat["mean"] * 100).round(2)
    print(by_cat[["count", "pass_pct"]])
